# 02 · Demonstrating superposition

**Paper:** the introduction figure (p. 2) and "Demonstrating Superposition", pp. 8–12.
**Experiments:** E00 `intro_pentagon`, E01 `basic_results`.

**Learning goals**

1. Know the setup exactly: sparse features, importance, the linear and ReLU output models, and
   the importance-weighted loss (pp. 9–10).
2. Reproduce the p. 2 figure and read it: dedicated dimensions → antipodal pairs → pentagon.
3. Understand why the linear model never uses superposition, and check it against its
   closed-form optimum.
4. Read the p. 11 visualization (WᵀW with bias, ‖Wᵢ‖ bars colored by superposition) and see
   superposition start with the least important features.

Runs use the `quick` profile. On a laptop CPU, E00 takes about a minute and the E01 section
below about two minutes; `--device auto` uses an Apple or NVIDIA GPU when there is one.

In [ ]:
# Runs locally and in Colab. In a fresh environment (e.g. Colab), install the package first.
# Replace <OWNER> with the GitHub owner of your copy of the repository.
try:
    import superposition  # noqa: F401
except ImportError:
    %pip install -q "git+https://github.com/<OWNER>/superposition.git"

In [ ]:
import pandas as pd
import torch
from IPython.display import Image, display

from superposition import metrics
from superposition.experiments.runner import run_experiment

pd.set_option("display.max_colwidth", 120)

## 1. The setup (pp. 9–10)

* **Data.** n features; xᵢ = 0 with probability S, else U[0, 1]; importance Iᵢ (here 0.9ⁱ).
* **Linear model:** x′ = WᵀWx + b. **ReLU output model:** x′ = ReLU(WᵀWx + b). They differ only
  in the final activation.
* **Loss:** L = Σₓ Σᵢ Iᵢ (xᵢ − x′ᵢ)².
* **Question:** with n = 5 features and only m = 2 dimensions, which features does each model
  keep, and how?

## 2. E00: five features in two dimensions (p. 2)

**Predict** before running: for S = 0, 0.8 and 0.9, how many features will the ReLU output
model represent, and in what arrangement? And the linear model?

In [ ]:
run = run_experiment("intro_pentagon", "quick", device="auto")
acceptance = run.read_acceptance()
print("overall:", acceptance["overall"], f"({acceptance['n_passed']}/{acceptance['n_graded']})")
pd.DataFrame(acceptance["criteria"])[["group", "id", "status", "value"]]

In [ ]:
display(Image(filename=str(run.figures_dir / "pentagon_arrows.png")))

Compare with p. 2: two orthogonal features when dense, two antipodal pairs at 80% sparsity,
a pentagon at 90%. The linear model keeps features 0 and 1 at every sparsity.

## 3. Why the linear model never uses superposition (p. 13)

For the linear model the loss is (roughly) Σᵢ Iᵢ(1 − ‖Wᵢ‖²)² + Σ_{i≠j} Iⱼ(Wⱼ·Wᵢ)²: a feature's
*benefit* against its *interference*. Interference always costs, so the best a linear model can
do is represent the m most important features orthogonally and predict the mean p/2 for the
rest. That gives an exact optimum, L* = Σ_{i≥m} Iᵢ (p/3 − p²/4).

**Predict:** how close do the trained linear models get?

In [ ]:
for row in acceptance["criteria"]:
    if row["group"] == "linear" and row["id"].endswith("loss_vs_closed_form"):
        print(row["id"], row["value"])

## 4. Inside the pentagon: negative biases filter positive interference

**Predict:** in the S = 0.9 pentagon, what sign are the biases, and are the norms above or
below 1?

In [ ]:
from superposition.experiments.common import GridResult

relu = GridResult.from_checkpoint(run.load_checkpoint("relu_output"))
W, b, loss = relu.best_weights(2)  # condition 2 is S = 0.9
print("norms ‖Wᵢ‖:", metrics.feature_norms(W[None])[0])
print("biases bᵢ :", b)
print("1-sparse readouts ReLU(WᵀW eᵢ + b), one row per active feature:")
print(torch.relu(metrics.gram(W[None])[0] + b).round(decimals=3))

Each row is the output when only one feature is active, with value 1. The diagonal is the
feature itself (≈ 1). Each feature's two non-neighbours on the pentagon (cos 144° < 0) get
exactly 0: the ReLU removes negative interference for free. Its two neighbours (cos 72°) still
receive about 0.15–0.2: the negative bias *reduces* positive interference but does not remove
it (with ‖W‖² ≈ 1.2 and b ≈ −0.21, a neighbour only leaks once 1.2 · 0.31 · x exceeds 0.21,
i.e. for x > 0.55). This residual is the "positive interference" of the p. 2 figure, the price
of fitting five features into two dimensions. These are the ingredients of the "1-sparse" loss
term L₁ the paper analyses on p. 13.

## 5. E01: twenty features in five dimensions (p. 11)

Now n = 20, m = 5, Iᵢ = 0.7ⁱ, and seven sparsities from 1 − S = 1 to 0.001. To keep this
notebook short we train just this model size with the library's grid trainer (the full E01
run, which adds n = 80, is `uv run superposition run basic_results`).

**Predict:** as sparsity increases, which features enter superposition first, the most or the
least important? At 1 − S = 0.001, roughly how many of the 20 features are represented?

In [ ]:
from superposition.config import TrainConfig
from superposition.data import geometric_importance
from superposition.experiments.common import train_sparsity_grid
from superposition.utils import resolve_device

probabilities = [1.0, 0.3, 0.1, 0.03, 0.01, 0.003, 0.001]
importance = geometric_importance(20, 0.7)
grids = {
    kind: train_sparsity_grid(
        kind,
        20,
        5,
        importance,
        probabilities,
        n_restarts=4,
        train_config=TrainConfig(),
        seed=0,
        device=resolve_device("auto"),
    )
    for kind in ("linear", "relu_output")
}
rows = []
for kind, grid in grids.items():
    for cond, p in enumerate(probabilities):
        W, _, loss = grid.best_weights(cond)
        rep = metrics.represented_mask(W[None])[0]
        score = metrics.superposition_score(W[None])[0]
        rows.append(
            {
                "model": kind,
                "1-S": p,
                "represented": int(rep.sum()),
                "in superposition": int((rep & (score > 0.05)).sum()),
                "loss": round(loss, 5),
            }
        )
pd.DataFrame(rows).pivot(
    index="1-S", columns="model", values=["represented", "in superposition"]
).sort_index(ascending=False)

In [ ]:
from superposition.plotting.weights import WeightsPanel, weights_grid


def panel(grid, cond, title):
    W, b, _ = grid.best_weights(cond)
    return WeightsPanel(
        title,
        metrics.gram(W[None])[0].numpy(),
        b.numpy(),
        metrics.feature_norms(W[None])[0].numpy(),
        metrics.superposition_score(W[None])[0].numpy(),
    )


panels = [panel(grids["linear"], 6, "1−S = 0.001")]
panels += [panel(grids["relu_output"], c, f"1−S = {p:g}") for c, p in enumerate(probabilities)]
fig = weights_grid(
    panels,
    suptitle="n = 20, m = 5 (compare p. 11)",
    group_labels=[(0, "Linear"), (1, "ReLU output model")],
)
fig

Read it like p. 11: the linear model keeps the top five features, orthogonal (dark bars). The
ReLU model matches it when dense; as sparsity rises, the *least* important represented features
turn bright first (superposition), often as antipodal pairs (blue off-diagonal cells in WᵀW),
while the most important features keep dedicated dimensions. At the sparsest settings almost
every feature is represented and in superposition, and the biases turn negative.

The full E01 run (both model sizes, acceptance criteria, published figures) is summarized in
`docs/REPRODUCTION.md`.

## Exercises

**1.** In section 5, find the first sparsity at which any feature is in superposition. Which
features are they, and is that consistent with "superposition appears first among the less
important features"?

<details><summary>Solution</summary>

```python
relu = grids["relu_output"]
for cond, p in enumerate(probabilities):
    W, _, _ = relu.best_weights(cond)
    rep = metrics.represented_mask(W[None])[0]
    sup = rep & (metrics.superposition_score(W[None])[0] > 0.05)
    if sup.any():
        print(p, torch.nonzero(sup).flatten().tolist()); break
```

Typically the first superposed features are the last few represented ones (e.g. features 3–6
at 1 − S = 0.3), while features 0–2 keep dedicated dimensions: the ordering claim of p. 11.
E01 checks this formally (criterion `relu_superposition_less_important_first`).
</details>

**2.** The ReLU model's loss at 1 − S = 0.001 is far below its loss at 1 − S = 1. Does that mean
superposition made it a better model? (Footnote 17.)

<details><summary>Solution</summary>

No. The two losses measure different tasks: the sparse task is easier because most inputs are
zero. Only models trained and evaluated on the same distribution can be compared. The fair
comparison is the ReLU model against the linear model at the *same* sparsity.
</details>

**3.** In E00's S = 0.9 pentagon the norms are about 1.1 and the biases about −0.2. Compute
‖Wᵢ‖² + bᵢ for each feature. What do you get, and why?

<details><summary>Solution</summary>

```python
print(metrics.feature_norms(W[None])[0] ** 2 + b)  # ≈ 1 for every feature
```

With only feature i active at value x, the output is ReLU(‖Wᵢ‖² x + bᵢ). The negative bias
filters interference but also shifts the feature's own readout down, so the model scales its
norm up until the largest input, x = 1, is reproduced exactly: ‖Wᵢ‖² + bᵢ ≈ 1. Small values of
x below −bᵢ/‖Wᵢ‖² ≈ 0.17 are clipped to 0, a price worth paying at high sparsity. The paper
describes the same compensation on p. 23.
</details>

**4.** Re-run E00 with a different seed (`run_experiment("intro_pentagon", "quick",
seed=1)`). Does it still pass? How many of the 8 restarts at S = 0.9 find all five features?

<details><summary>Solution</summary>

Look at the `restart_outcomes` INFO row. Typically most, but not all, restarts reach the
pentagon; the rest settle for four features. This is the optimization difficulty of 2D hidden
spaces that the paper mentions on p. 23, and the reason E00 keeps the best of 8 restarts.
</details>

**5.** Why does E00 evaluate its final losses on 100,000 fresh samples instead of 20,000?
(Look at the `small_sample_eval_noise` INFO row.)

<details><summary>Solution</summary>

At S = 0.9 most samples have zero or one active feature, and the loss is dominated by rarer
multi-feature samples, so a 20,000-sample estimate has about 3–4% relative noise. The linear
closed-form check needs better than 3% accuracy, and choosing the best restart by a noisy
estimate biases the chosen loss downwards (a winner's curse).
</details>